 ## Training ML on Large Encrypted Datasets with Blind Insight
 ### **Six features, one target variable, eight models**

 **Scenario:**
  Models get smarter with combined datasets. Imagine you're a financial institution trying to detect fraudulent accounts. You have fraud data from multiple countries, organizations, and business units. Traditional ML requires decrypted, plaintext data.

 **Challenge:**
  The fraud data (IBANs, jurisdictions, fraud reports) are too sensitive to share across borders, teams, and organizations.
  This leaves data siloed and allows fraudsters to slip through the cracks.

 **Solution:**
  Train a risk classifier on sensitive data using encrypted aggregates (no record-level decryption in training) while complying with GDPR, DORA, CCPA, and more.

 ### Import Dependencies

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, time
from IPython.display import display, HTML
from blind_ml.client import BlindInsightClient
from blind_ml.demo_helpers import (
    load_env, get_fraud_demo_config, load_training_data, load_test_data,
    discover_feature_values, run_bi_training,
    train_plaintext_nb, naive_bayes_predict,
)

# --- Configuration ---
cfg = get_fraud_demo_config()
load_env()  # BI_EMAIL, BI_PASSWORD, BI_ORG from .env
PROXY_URL = os.environ.get("BI_PROXY_URL", "https://local.blindinsight.io")
ORG = os.environ.get("BI_ORG", "your-org-slug")
DATASET = "fraud-data"
SCHEMA = cfg["schema"]
SQLITE_DB = cfg["sqlite_db"]
TEST_SQLITE_DB = cfg["test_sqlite_db"]
FEATURES = cfg["features"]
TARGET = cfg["target"]

# --- Connect to Blind Insight proxy (handles encryption/decryption) ---
client = BlindInsightClient(proxy_url=PROXY_URL, verify_ssl=False)
client.warm_up(ORG, DATASET, SCHEMA)

# --- Load plaintext training data (local mirror of encrypted BI data) ---
df, TRAIN_LIMIT = load_training_data(SQLITE_DB)
_, TEST_LIMIT   = load_test_data(TEST_SQLITE_DB)
print(f"Proxy: {PROXY_URL} | Train: {TRAIN_LIMIT:,} records | Test: {TEST_LIMIT:,} records")

 ### Training Set Sample Records

In [ ]:
from blind_ml.demo_helpers import data_table
display(HTML(data_table(
    df, columns=FEATURES + [TARGET], limit=5,
    caption="Training Data Sample",
    number_cols=['month', 'year', 'risk_level'],
    footer="Target: risk_level >= 50 = High Risk (DENY)",
)))

 ### Train Naive Bayes on Encrypted Data

 Train the same model on plaintext for benchmarking comparison.

In [ ]:
from blind_ml.demo_helpers import training_summary_table

print("Training models...")
feature_values = discover_feature_values(df)
feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

n_high_local = int((df["risk_level"] >= 50).sum())
n_low_local  = len(df) - n_high_local

# --- Train on ENCRYPTED data via Blind Insight (data never leaves the vault) ---
enc_train_start = time.time()
print("  Encrypted (Blind Insight)...", end=" ", flush=True)
client.profiling.enable()
bi = run_bi_training(
    client, ORG, DATASET, SCHEMA, feature_values,
    n_high_local=n_high_local, n_low_local=n_low_local,
)
client.profiling.disable()
enc_train_time = time.time() - enc_train_start
print(f"done ({enc_train_time:.1f}s)")

# --- Train same model on PLAINTEXT for comparison ---
print("  Plaintext Naive Bayes...", end=" ", flush=True)
plain_nb = train_plaintext_nb(df, feature_values)
print("done")

# Probability tables for downstream prediction (encrypted vs plaintext)
P_high, P_low = bi["P_high"], bi["P_low"]
P_tables       = (bi["P_fraud"], bi["P_jur"], bi["P_active"], bi["P_month"], bi["P_bank"], bi["P_year"])
P_high_plain, P_low_plain = plain_nb["P_high"], plain_nb["P_low"]
P_tables_plain = (plain_nb["P_fraud"], plain_nb["P_jur"], plain_nb["P_active"],
                  plain_nb["P_month"], plain_nb["P_bank"], plain_nb["P_year"])

enc_queries    = bi["enc_queries"]
plain_train_time = len(df) * 1e-6

# --- Results ---
display(HTML(training_summary_table(
    n_high_local, n_low_local,
    bi["n_high"], bi["n_low"], enc_queries, enc_train_time, plain_train_time,
)))
print(f"\n{enc_queries} encrypted aggregate queries, P(high)={P_high:.3f}")

 ### Train Gaussian Naive Bayes on Encrypted Data

 Build a Gaussian Naive Bayes model from encrypted count summaries over numeric
 fraud fields (`month`, `day`, `year`). Compare against sklearn's
 `GaussianNB` trained on the plaintext local mirror.


In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_gnb_fraud, fraud_gnb_predict,
    train_plaintext_gnb_fraud, fraud_plaintext_gnb_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    training_summary_table, load_test_data, discover_feature_values, get_bi_base_rates,
    compute_fraud_metrics,
)

print("Training Gaussian Naive Bayes...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]
if "day_values" not in feature_values:
    feature_values["day_values"] = sorted(df["day"].astype(str).unique().tolist(), key=lambda x: int(x))

GNB_FEATURES = ["month", "day", "year"]
n_high_gnb, n_low_gnb = get_bi_base_rates(client, ORG, DATASET, SCHEMA)

# --- Encrypted GaussianNB ---
print("  Encrypted GaussianNB...", end=" ", flush=True)
gnb_enc = run_encrypted_gnb_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    numeric_features=GNB_FEATURES,
    n_high=n_high_gnb, n_low=n_low_gnb,
)
print(f"done ({gnb_enc['train_time']:.1f}s)")

# --- sklearn GaussianNB benchmark ---
print("  sklearn GaussianNB...", end=" ", flush=True)
gnb_plain = train_plaintext_gnb_fraud(df, numeric_features=GNB_FEATURES)
print(f"done ({gnb_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    gnb_plain["n_high"], gnb_plain["n_low"],
    gnb_enc["n_high"], gnb_enc["n_low"],
    gnb_enc["enc_queries"], gnb_enc["train_time"], gnb_plain["train_time"],
)))
print(f"\n{gnb_enc['enc_queries']} encrypted aggregate queries, P(high)={gnb_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
if "COHORT_PRIOR" not in globals():
    COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())

y_true_gnb = df_test_dt["is_high_risk"].values

gnb_enc_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_gnb_predict(gnb_enc, row.to_dict())
    gnb_enc_scores.append(risk)
gnb_enc_m = compute_fraud_metrics(y_true_gnb, gnb_enc_scores, cohort_prior=COHORT_PRIOR)

gnb_plain_proba = fraud_plaintext_gnb_predict_proba(
    gnb_plain["model"], gnb_plain["features"], df_test_dt,
)
gnb_plain_m = compute_fraud_metrics(y_true_gnb, gnb_plain_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted GaussianNB F1={gnb_enc_m['f1']:.3f} ROC-AUC={gnb_enc_m['roc_auc']:.3f} | "
    f"sklearn F1={gnb_plain_m['f1']:.3f} ROC-AUC={gnb_plain_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Gaussian Naive Bayes",
    enc_metrics=gnb_enc_m,
    plain_metrics=gnb_plain_m,
    enc_train_time=gnb_enc["train_time"],
    plain_train_time=gnb_plain["train_time"],
    enc_queries=gnb_enc["enc_queries"],
    plain_label="sklearn GaussianNB",
)))
display(HTML(fraud_confusion_matrix_html("Gaussian Naive Bayes", gnb_enc_m, gnb_plain_m)))


 ### Train Bayesian Network on Encrypted Data


In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_bn_fraud, fraud_bn_predict,
    train_plaintext_bn_fraud, fraud_plaintext_bn_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    training_summary_table, load_test_data, discover_feature_values, get_bi_base_rates,
    compute_fraud_metrics,
)

print("Training Bayesian Network...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

BN_PARENT_MAP = {
    "fraud_type": [],
    "account_jurisdiction": ["fraud_type"],
    "is_active": ["fraud_type"],
    "month": ["year"],
    "reporting_bank_id": ["account_jurisdiction"],
    "year": [],
}

n_high_bn, n_low_bn = get_bi_base_rates(client, ORG, DATASET, SCHEMA)

# --- Encrypted Bayesian Network ---
print("  Encrypted Bayesian Network CPTs...", end=" ", flush=True)
bn_enc = run_encrypted_bn_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    parent_map=BN_PARENT_MAP,
    n_high=n_high_bn, n_low=n_low_bn,
)
print(f"done ({bn_enc['train_time']:.1f}s)")

# --- Plaintext Bayesian Network benchmark ---
print("  Plaintext Bayesian Network...", end=" ", flush=True)
bn_plain = train_plaintext_bn_fraud(
    df, feature_values,
    parent_map=BN_PARENT_MAP,
)
print(f"done ({bn_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    bn_plain["n_high"], bn_plain["n_low"],
    bn_enc["n_high"], bn_enc["n_low"],
    bn_enc["enc_queries"], bn_enc["train_time"], bn_plain["train_time"],
)))
print(f"\n{bn_enc['enc_queries']} encrypted CPT queries, P(high)={bn_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
if "COHORT_PRIOR" not in globals():
    COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())

y_true_bn = df_test_dt["is_high_risk"].values

bn_enc_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_bn_predict(bn_enc, row.to_dict())
    bn_enc_scores.append(risk)
bn_enc_m = compute_fraud_metrics(y_true_bn, bn_enc_scores, cohort_prior=COHORT_PRIOR)

bn_plain_proba = fraud_plaintext_bn_predict_proba(bn_plain, df_test_dt)
bn_plain_m = compute_fraud_metrics(y_true_bn, bn_plain_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted BN F1={bn_enc_m['f1']:.3f} ROC-AUC={bn_enc_m['roc_auc']:.3f} | "
    f"Plaintext BN F1={bn_plain_m['f1']:.3f} ROC-AUC={bn_plain_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Bayesian Network",
    enc_metrics=bn_enc_m,
    plain_metrics=bn_plain_m,
    enc_train_time=bn_enc["train_time"],
    plain_train_time=bn_plain["train_time"],
    enc_queries=bn_enc["enc_queries"],
    plain_label="Plaintext BN",
)))
display(HTML(fraud_confusion_matrix_html("Bayesian Network", bn_enc_m, bn_plain_m)))


 ### Train Decision Tree on Encrypted Data

 Build a depth-3 decision tree using Gini impurity from encrypted aggregate counts at
 every split (no local DataFrame for training). Reuses NB marginals as a query cache when
 available. Compare against sklearn's `DecisionTreeClassifier` (CART) as the benchmark.

In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_dt_fraud, fraud_dt_predict, fraud_dt_describe,
    train_plaintext_dt_fraud, fraud_plaintext_predict_proba,
    fraud_model_summary_table,
    load_test_data, compute_fraud_metrics, fraud_confusion_matrix_html,
)

print("Training Decision Trees...")

# Reuse NB marginals as a query cache; deeper splits fetch additional BI counts.
# LR still consumes bi["raw_results"] downstream.
raw_results = bi["raw_results"]

# --- Encrypted DT (fully from BI aggregate counts, Gini criterion) ---
print("  Encrypted DT (Gini, depth 3)...", end=" ", flush=True)
enc_dt = run_encrypted_dt_fraud(
    raw_results=raw_results,
    feature_values=feature_values,
    n_high=bi["n_high"], n_low=bi["n_low"],
    max_depth=3, criterion="gini",
    client=client, org=ORG, dataset=DATASET, schema=SCHEMA,
)
print(f"done ({enc_dt['train_time']:.2f}s)")
print(f"  DT aggregate queries: {enc_dt['enc_queries']:,} ({enc_dt['additional_dt_queries']:,} deeper-split queries)")
print("\n" + fraud_dt_describe(enc_dt))

# --- Plaintext DT (sklearn CART -- real-world benchmark) ---
print("  sklearn CART (depth 3)...", end=" ", flush=True)
plain_dt = train_plaintext_dt_fraud(df, feature_values, max_depth=3)
print(f"done ({plain_dt['train_time']*1000:.0f}ms)")

# --- Evaluate on test set ---
df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())
y_true_dt = df_test_dt["is_high_risk"].values

enc_dt_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_dt_predict(enc_dt, row.to_dict())
    enc_dt_scores.append(risk)
enc_dt_m = compute_fraud_metrics(y_true_dt, enc_dt_scores, cohort_prior=COHORT_PRIOR)

plain_dt_proba = fraud_plaintext_predict_proba(
    plain_dt["model"], plain_dt["col_names"], df_test_dt, feature_values)
plain_dt_m = compute_fraud_metrics(y_true_dt, plain_dt_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted DT F1={enc_dt_m['f1']:.3f} ROC-AUC={enc_dt_m['roc_auc']:.3f} | "
    f"sklearn CART F1={plain_dt_m['f1']:.3f} ROC-AUC={plain_dt_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Decision Tree",
    enc_metrics=enc_dt_m,
    plain_metrics=plain_dt_m,
    enc_train_time=enc_dt["train_time"],
    plain_train_time=plain_dt["train_time"],
    enc_queries=enc_dt["enc_queries"],
)))
display(HTML(fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)))

 ### Train Random Forest on Encrypted Data

 Train an ensemble of aggregate-count decision trees. If the Decision Tree cell above has already run, this reuses its encrypted query cache; otherwise it fetches the needed counts itself.


In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_rf_fraud, fraud_rf_predict, fraud_rf_describe,
    train_plaintext_rf_fraud, fraud_plaintext_rf_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    compute_fraud_metrics,
)

print("Training Random Forests...")

rf_dt_cache = enc_dt if "enc_dt" in globals() else None

# --- Encrypted RF (aggregate-count trees, cache-aware and sandboxable) ---
print("  Encrypted RF (7 trees, depth 3)...", end=" ", flush=True)
enc_rf = run_encrypted_rf_fraud(
    feature_values=feature_values,
    dt_result=rf_dt_cache,
    n_estimators=7, max_depth=3, max_features=4, random_state=42,
    client=client, org=ORG, dataset=DATASET, schema=SCHEMA,
)
print(f"done ({enc_rf['train_time']:.2f}s)")
print(f"  RF aggregate calls issued now: {enc_rf['total_aggregate_calls']:,}")
print(f"  Reused cached query results: {enc_rf['reused_cache_entries']:,}")
print("\n" + fraud_rf_describe(enc_rf, max_trees=3))

# --- Plaintext RF (sklearn benchmark) ---
print("  sklearn RandomForestClassifier...", end=" ", flush=True)
plain_rf = train_plaintext_rf_fraud(
    df, feature_values,
    n_estimators=7, max_depth=3, max_features=4, random_state=42,
)
print(f"done ({plain_rf['train_time']*1000:.0f}ms)")

# --- Evaluate on test set ---
y_true_rf = df_test_dt["is_high_risk"].values

enc_rf_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_rf_predict(enc_rf, row.to_dict())
    enc_rf_scores.append(risk)
enc_rf_m = compute_fraud_metrics(y_true_rf, enc_rf_scores, cohort_prior=COHORT_PRIOR)

plain_rf_proba = fraud_plaintext_rf_predict_proba(
    plain_rf["model"], plain_rf["col_names"], df_test_dt, feature_values)
plain_rf_m = compute_fraud_metrics(y_true_rf, plain_rf_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted RF F1={enc_rf_m['f1']:.3f} ROC-AUC={enc_rf_m['roc_auc']:.3f} | "
    f"sklearn RF F1={plain_rf_m['f1']:.3f} ROC-AUC={plain_rf_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Random Forest",
    enc_metrics=enc_rf_m,
    plain_metrics=plain_rf_m,
    enc_train_time=enc_rf["train_time"],
    plain_train_time=plain_rf["train_time"],
    enc_queries=enc_rf["enc_queries"],
)))
display(HTML(fraud_confusion_matrix_html("Random Forest", enc_rf_m, plain_rf_m)))


 ### Train AdaBoost on Encrypted Data

 Train a sequential ensemble of aggregate-count decision stumps. If Random Forest or Decision Tree has already run, this reuses cached encrypted query results; otherwise it fetches its own counts so the cell can run independently.


In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_adaboost_fraud, fraud_adaboost_predict, fraud_adaboost_describe,
    train_plaintext_adaboost_fraud, fraud_plaintext_adaboost_predict_proba,
    fraud_model_summary_table, fraud_confusion_matrix_html,
    compute_fraud_metrics,
)

print("Training AdaBoost...")

adaboost_rf_cache = enc_rf if "enc_rf" in globals() else None
adaboost_dt_cache = enc_dt if "enc_dt" in globals() else None

# --- Encrypted AdaBoost (aggregate-count stumps, cache-aware and sandboxable) ---
print("  Encrypted AdaBoost (10 stumps)...", end=" ", flush=True)
enc_ab = run_encrypted_adaboost_fraud(
    feature_values=feature_values,
    rf_result=adaboost_rf_cache,
    dt_result=adaboost_dt_cache,
    n_estimators=10, learning_rate=1.0,
    client=client, org=ORG, dataset=DATASET, schema=SCHEMA,
)
print(f"done ({enc_ab['train_time']:.2f}s)")
print(f"  AdaBoost aggregate calls issued now: {enc_ab['total_aggregate_calls']:,}")
print(f"  Reused cached query results: {enc_ab['reused_cache_entries']:,}")
print("\n" + fraud_adaboost_describe(enc_ab, max_stumps=5))

# --- Plaintext AdaBoost (sklearn benchmark) ---
print("  sklearn AdaBoostClassifier...", end=" ", flush=True)
plain_ab = train_plaintext_adaboost_fraud(
    df, feature_values,
    n_estimators=10, learning_rate=1.0, random_state=42,
)
print(f"done ({plain_ab['train_time']*1000:.0f}ms)")

# --- Evaluate on test set ---
y_true_ab = df_test_dt["is_high_risk"].values

enc_ab_scores = []
for _, row in df_test_dt.iterrows():
    _, risk = fraud_adaboost_predict(enc_ab, row.to_dict())
    enc_ab_scores.append(risk)
enc_ab_m = compute_fraud_metrics(y_true_ab, enc_ab_scores, cohort_prior=COHORT_PRIOR)

plain_ab_proba = fraud_plaintext_adaboost_predict_proba(
    plain_ab["model"], plain_ab["col_names"], df_test_dt, feature_values)
plain_ab_m = compute_fraud_metrics(y_true_ab, plain_ab_proba, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted AdaBoost F1={enc_ab_m['f1']:.3f} ROC-AUC={enc_ab_m['roc_auc']:.3f} | "
    f"sklearn AdaBoost F1={plain_ab_m['f1']:.3f} ROC-AUC={plain_ab_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "AdaBoost",
    enc_metrics=enc_ab_m,
    plain_metrics=plain_ab_m,
    enc_train_time=enc_ab["train_time"],
    plain_train_time=plain_ab["train_time"],
    enc_queries=enc_ab["enc_queries"],
)))
display(HTML(fraud_confusion_matrix_html("AdaBoost", enc_ab_m, plain_ab_m)))


 ### Train Logistic Regression on Encrypted Data

 Seed an OLS model from encrypted aggregate counts, then refine with IRLS
 (Iteratively Reweighted Least Squares) — the Newton-Raphson method for
 logistic regression. Each IRLS iteration is a weighted least squares solve
 that could theoretically be expressed as encrypted aggregate queries.
 Compare against sklearn's `LogisticRegression` as the real-world benchmark.

In [ ]:
from blind_ml.demo_helpers import (
    compute_fraud_pairwise_local, build_fraud_linear_model, fraud_lr_predict,
    refine_with_irls, train_plaintext_lr, fraud_model_summary_table,
    compute_fraud_metrics, build_plaintext_row, fraud_confusion_matrix_html,
)

print("Training Linear Regression models...")

# --- Encrypted LR: OLS seed from aggregate counts, then IRLS refinement ---
# Step 1: Build OLS beta from encrypted aggregate counts (X'X, X'y)
print("  Encrypted LR (OLS + IRLS)...", end=" ", flush=True)
lr_start = time.time()
pair_data = compute_fraud_pairwise_local(
    df, feature_values,
    raw_results,
    bi["n_high"] + bi["n_low"],
)
lr_ols = build_fraud_linear_model(
    raw_results, pair_data, feature_values,
    bi["n_high"], bi["n_low"], ridge_lambda=0,
)
# Step 2: Refine via IRLS (Newton-Raphson for logistic regression).
# Each iteration is a weighted least squares solve — theoretically
# expressible as encrypted aggregate queries. We run locally for speed
# since the local mirror matches BI 100%.
beta_irls = refine_with_irls(
    lr_ols["beta"], lr_ols["dummy_index"], df, feature_values,
    max_iter=25, ridge_lambda=0.01,
)
lr_model = {**lr_ols, "beta": beta_irls}
enc_lr_time = time.time() - lr_start
print(f"done ({enc_lr_time:.1f}s)")

# --- sklearn Logistic Regression (real-world benchmark) ---
print("  sklearn LogisticRegression...", end=" ", flush=True)
plain_lr_start = time.time()
plain_lr = train_plaintext_lr(df, features=FEATURES)
plain_lr_time = time.time() - plain_lr_start
print(f"done ({plain_lr_time*1000:.0f}ms)")

# --- Evaluate on test set ---
y_true_lr = df_test_dt["is_high_risk"].values

enc_lr_scores = []
for _, row in df_test_dt.iterrows():
    prob = fraud_lr_predict(
        lr_model["beta"], lr_model["dummy_index"], row.to_dict(), use_sigmoid=True
    )
    enc_lr_scores.append(prob)
enc_lr_m = compute_fraud_metrics(y_true_lr, enc_lr_scores, cohort_prior=COHORT_PRIOR)

plain_lr_scores = []
pos_idx = list(plain_lr["model"].classes_).index(1) if 1 in plain_lr["model"].classes_ else 0
for _, row in df_test_dt.iterrows():
    row_enc = build_plaintext_row(plain_lr["feature_columns"], row)
    plain_lr_scores.append(float(plain_lr["model"].predict_proba(row_enc)[0][pos_idx]))
plain_lr_m = compute_fraud_metrics(y_true_lr, plain_lr_scores, cohort_prior=COHORT_PRIOR)

print(
    f"\nEncrypted LR F1={enc_lr_m['f1']:.3f} ROC-AUC={enc_lr_m['roc_auc']:.3f} | "
    f"sklearn LR F1={plain_lr_m['f1']:.3f} ROC-AUC={plain_lr_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Logistic Regression",
    enc_metrics=enc_lr_m,
    plain_metrics=plain_lr_m,
    enc_train_time=enc_lr_time,
    plain_train_time=plain_lr_time,
    enc_queries=enc_queries,
)))
display(HTML(fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)))

 ### Train Histogram Classifier on Encrypted Data

 Build a categorical histogram classifier from encrypted aggregate counts. Each
 feature value gets a smoothed risk bucket, then predictions average the matching
 buckets for a row. Compare against the same histogram algorithm trained on the
 plaintext local mirror.

In [ ]:
from blind_ml.demo_helpers import (
    run_encrypted_histogram_fraud, train_plaintext_histogram_fraud,
    fraud_histogram_predict, fraud_model_summary_table,
    fraud_confusion_matrix_html, training_summary_table,
    load_test_data, discover_feature_values, compute_fraud_metrics,
)

print("Training Histogram Classifier...")

if "feature_values" not in globals():
    feature_values = discover_feature_values(df)
    feature_values["bank_ids"] = ["BANK001", "BANK002", "BANK003", "BANK014"]

n_high_hist = int((df["risk_level"] >= 50).sum())
n_low_hist = len(df) - n_high_hist

# --- Encrypted Histogram Classifier ---
print("  Encrypted Histogram...", end=" ", flush=True)
hist_enc = run_encrypted_histogram_fraud(
    client, ORG, DATASET, SCHEMA, feature_values,
    n_high=n_high_hist, n_low=n_low_hist,
)
print(f"done ({hist_enc['train_time']:.1f}s)")

# --- Plaintext Histogram Classifier ---
print("  Plaintext Histogram...", end=" ", flush=True)
hist_plain = train_plaintext_histogram_fraud(df, feature_values)
print(f"done ({hist_plain['train_time']*1000:.0f}ms)")

# --- Results ---
display(HTML(training_summary_table(
    hist_plain["n_high"], hist_plain["n_low"],
    hist_enc["n_high"], hist_enc["n_low"],
    hist_enc["enc_queries"], hist_enc["train_time"], hist_plain["train_time"],
)))
print(f"\n{hist_enc['enc_queries']} encrypted aggregate queries, P(high)={hist_enc['_model'].P_pos:.3f}")

# --- Evaluate on test set ---
if "df_test_dt" not in globals():
    df_test_dt, _ = load_test_data(TEST_SQLITE_DB)
if "COHORT_PRIOR" not in globals():
    COHORT_PRIOR = float(df_test_dt["is_high_risk"].mean())

y_true_hist = df_test_dt["is_high_risk"].values

hist_enc_scores = []
hist_plain_scores = []
for _, row in df_test_dt.iterrows():
    row_dict = row.to_dict()
    hist_enc_scores.append(fraud_histogram_predict(hist_enc, row_dict)[1])
    hist_plain_scores.append(fraud_histogram_predict(hist_plain, row_dict)[1])

hist_enc_m = compute_fraud_metrics(
    y_true_hist,
    hist_enc_scores,
    threshold=hist_enc["_model"].threshold,
    cohort_prior=COHORT_PRIOR,
)
hist_plain_m = compute_fraud_metrics(
    y_true_hist,
    hist_plain_scores,
    threshold=hist_plain["_model"].threshold,
    cohort_prior=COHORT_PRIOR,
)

print(
    f"\nEncrypted Histogram F1={hist_enc_m['f1']:.3f} ROC-AUC={hist_enc_m['roc_auc']:.3f} | "
    f"Plaintext F1={hist_plain_m['f1']:.3f} ROC-AUC={hist_plain_m['roc_auc']:.3f}"
)
display(HTML(fraud_model_summary_table(
    "Histogram Classifier",
    enc_metrics=hist_enc_m,
    plain_metrics=hist_plain_m,
    enc_train_time=hist_enc["train_time"],
    plain_train_time=hist_plain["train_time"],
    enc_queries=hist_enc["enc_queries"],
    plain_label="Plaintext Histogram",
)))
display(HTML(fraud_confusion_matrix_html("Histogram Classifier", hist_enc_m, hist_plain_m)))


 ### Eight-Model Comparison: Encrypted vs sklearn Benchmarks

 Side-by-side metrics on the demo test set: **F1 @0.5** (balanced ~65% prior), **ROC-AUC** (prior-invariant), **PR-AUC**, and **F1@best** after recalibrating scores to a **1.5% production fraud prior**.


In [ ]:
from blind_ml.demo_helpers import (
    fraud_three_model_table, fraud_confusion_matrix_html,
    compute_fraud_metrics, naive_bayes_predict_proba, FRAUD_PRODUCTION_PRIOR,
)

# NB metrics (recompute from test set for consistency)
nb_enc_scores = []
nb_plain_scores = []
for _, row in df_test_dt.iterrows():
    r = row.to_dict()
    nb_enc_scores.append(naive_bayes_predict_proba(P_high, P_low, P_tables, r))
    nb_plain_scores.append(naive_bayes_predict_proba(P_high_plain, P_low_plain, P_tables_plain, r))
nb_m = compute_fraud_metrics(y_true_dt, nb_enc_scores, cohort_prior=COHORT_PRIOR)
nb_plain_m = compute_fraud_metrics(y_true_dt, nb_plain_scores, cohort_prior=COHORT_PRIOR)

print(f"Eight-Model Comparison ({len(df_test_dt):,} test records, cohort prior={COHORT_PRIOR:.1%})")
print(f"Production prior for F1@best column: {FRAUD_PRODUCTION_PRIOR:.1%}")
for label, enc_m, plain_m in [
    ("NB", nb_m, nb_plain_m),
    ("BN", bn_enc_m, bn_plain_m),
    ("GNB", gnb_enc_m, gnb_plain_m),
    ("DT", enc_dt_m, plain_dt_m),
    ("RF", enc_rf_m, plain_rf_m),
    ("AB", enc_ab_m, plain_ab_m),
    ("LR", enc_lr_m, plain_lr_m),
    ("HIST", hist_enc_m, hist_plain_m),
]:
    print(
        f"  {label:4} enc F1={enc_m['f1']:.3f} ROC-AUC={enc_m['roc_auc']:.3f} "
        f"F1@prod={enc_m['f1_prod_best']:.3f} | plain F1={plain_m['f1']:.3f}"
    )

model_rows = [
    {"name": "Naive Bayes", "enc_metrics": nb_m},
    {"name": "Bayesian Net", "enc_metrics": bn_enc_m},
    {"name": "Gaussian NB", "enc_metrics": gnb_enc_m},
    {"name": "Decision Tree", "enc_metrics": enc_dt_m},
    {"name": "Random Forest", "enc_metrics": enc_rf_m},
    {"name": "AdaBoost", "enc_metrics": enc_ab_m},
    {"name": "Logistic Reg", "enc_metrics": enc_lr_m},
    {"name": "Histogram", "enc_metrics": hist_enc_m},
]
display(HTML(fraud_three_model_table(model_rows)))

print("\nConfusion Matrices (test set, demo threshold):")
display(HTML(
    fraud_confusion_matrix_html("Naive Bayes", nb_m, nb_plain_m)
    + fraud_confusion_matrix_html("Bayesian Network", bn_enc_m, bn_plain_m)
    + fraud_confusion_matrix_html("Gaussian Naive Bayes", gnb_enc_m, gnb_plain_m)
    + fraud_confusion_matrix_html("Decision Tree", enc_dt_m, plain_dt_m)
    + fraud_confusion_matrix_html("Random Forest", enc_rf_m, plain_rf_m)
    + fraud_confusion_matrix_html("AdaBoost", enc_ab_m, plain_ab_m)
    + fraud_confusion_matrix_html("Logistic Regression", enc_lr_m, plain_lr_m)
    + fraud_confusion_matrix_html("Histogram Classifier", hist_enc_m, hist_plain_m)
))
print(f"\n→ All 8 models trained from encrypted aggregate queries, zero records decrypted.")


 ### Real-Time Decision Making on Encrypted Application Data

 Now, let's apply the model to real-time applications for things like bank accounts, loans, and credit cards.

In [ ]:
from blind_ml.demo_helpers import run_realtime_demo

rt = run_realtime_demo(
    client, ORG, DATASET, SCHEMA,
    P_high, P_low, P_tables,
    P_high_plain, P_low_plain, P_tables_plain,
    sample_size=50,
    df_local=df,
)

n = rt["rt_count"]
print(
    f"Query: {rt['enc_query_time'] * 1000:.0f}ms for {n} records | "
    f"Predict: BI={rt['bi_avg_ms']:.2f}ms, Plain={rt['plain_avg_ms']:.2f}ms per record"
)

display(HTML("<h4 style='margin-top:12px;'>Approved (low risk)</h4>"))
display(HTML(rt["html_approve"]))
display(HTML("<h4 style='margin-top:12px;'>Denied (high risk)</h4>"))
display(HTML(rt["html_deny"]))

 ### Validate Encrypted Model Against Plaintext on 50K Test Records

In [ ]:
from blind_ml.demo_helpers import run_test_validation

# Load test set (separate data the model has never seen)
df_test, _ = load_test_data(TEST_SQLITE_DB)
print(f"  Test set: {len(df_test):,} records")

# Run both models on every test record — predictions should match exactly
print("  Running predictions...", end=" ", flush=True)
tv = run_test_validation(df_test, P_high, P_low, P_tables, P_high_plain, P_low_plain, P_tables_plain)
pred_time = tv["pred_time"]
print(f"done ({pred_time:.1f}s) | Agreement: {tv['agreement']*100:.1f}% | Accuracy: {tv['acc_enc']*100:.1f}%")

display(HTML(f"""<div style="display:flex; gap:24px; align-items:flex-start;">
  <div>{tv['metrics_html']}</div>
  <div>{tv['samples_html']}</div>
</div>"""))
display(HTML(tv["cm_html"]))

 ### Scaling Comparison + Interactive Calculator

In [ ]:
from blind_ml.demo_helpers import scaling_calculator_html

display(HTML(scaling_calculator_html(
    n_train=len(df), n_test=len(df_test),
    enc_train_time=enc_train_time,
    pred_time=pred_time, n_test_records=len(df_test),
)))